# Scenario I [UNSW]

Figures and results for scenario II with UNSW gear crack data. Model trained with AGFD & MCC5 data


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from einops import rearrange, reduce, repeat
from matplotlib import pyplot as plt

from src.data.preprocessing_individual import get_important_frequencies
from src.data.signal_analysis import fold
from src.models.models import setup_model
from src.utils.init import setup_device

## Setup

### Plotting


### Config


In [ ]:
# SETUP

UNSW_teeth = 27
AGFD_teeth = 15
MCC5_teeth = 36

embedding_columns = ["x", "y"]

embedding_specs = {
    "normalize_embeddings": False,
    "average_anchors": True,
    # "average_batch": False,  # XXX
    "average_batch": True,
    "drop_anchor_outliers": 0,
    "drop_batch_outliers": 5,
}

config = {
    "use_difference": True,
    "gear_harmonics_to_include": 8,
    # "gear_sideband_block_size": 2,
    # "gear_sideband_block_size": 7, # XXX
    "gear_sideband_block_size": 5,
    "embedding_len": 2,
    "model": "embedding",
    "backbone": "MlpMixer",
    "model_args": {
        "intermediate_L2": False,
        # NORMAL
        "num_layers": 4,
        "hidden_dim": 21,
        "tokens_mlp_dim": 16,
        "channels_mlp_dim": 42,
        # DOUBLE
        # "num_layers": 4,
        # "hidden_dim": 52,
        # "tokens_mlp_dim": 32,
        # "channels_mlp_dim": 84,
        # TUNI
        # "num_layers": 4,
        # "hidden_dim": 18,
        # "tokens_mlp_dim": 16,
        # "channels_mlp_dim": 18,
    },
    "dropout": 0.0,
    "loss": "triplet",
    "loss_args": {
        "margin": 1.4,  # Doesn't matter during inference
        "miner": False,  # Doesn't matter during inference
    },
    "similarity": "euclidean",  # Doesn't matter during inference
    "lpnorm_embeddings_training": False,  # All of these are False to do them manually
    "center_embeddings_inference": False,
    "lpnorm_embeddings_inference": False,
}

device = setup_device(override="cpu")

model = setup_model(config, device)
model.eval()

# Trained model

model_weight_dir = Path.cwd().resolve() / "model_weights"
ensemble_weight_dir_name = "<FILL_ME>"
ensemble_repeat_num = 1
ensemble_checkpoint_num = 9  # XXX
# ensemble_checkpoint_num = 49  # XXX

### Helpers


In [ ]:
def remove_anchor_outliers(embeddings, num_outliers=5):
    filtered_embeddings = []
    # Calculate mean embedding
    mean_embedding = torch.mean(embeddings, dim=1, keepdim=True)

    # Calculate distances from each anchor to the mean
    distances = torch.norm(embeddings - mean_embedding, dim=2)

    # Find indices of closest anchors
    for i in range(distances.shape[0]):
        _, closest_indices = torch.topk(
            distances[i], distances.shape[1] - num_outliers, largest=False
        )
        filtered_embeddings.append(embeddings[i][closest_indices, :])

    filtered_embeddings = torch.stack(filtered_embeddings, dim=0)

    return filtered_embeddings


def remove_batch_outliers(embeddings, num_outliers=5):
    # Calculate mean embedding
    mean_embedding = torch.mean(embeddings, dim=0, keepdim=True)

    # Calculate distances from each anchor to the mean
    distances = torch.norm(embeddings - mean_embedding, dim=1)

    # Find indices of closest anchors
    _, closest_indices = torch.topk(
        distances, distances.shape[0] - num_outliers, largest=False
    )
    return embeddings[closest_indices, :]


def embed(
    samples,
    embedding_model,
    normalize_embeddings=False,
    average_anchors=False,
    average_batch=False,
    drop_anchor_outliers=0,
    drop_batch_outliers=0,
):
    assert not average_batch or average_anchors, (
        "Must average anchors if averaging batch"
    )

    assert drop_batch_outliers == 0 or average_anchors, (
        "Must average anchors if dropping batch outliers"
    )

    samples = torch.tensor(samples, device=device)

    embeddings = embedding_model(samples)

    if drop_anchor_outliers > 0:
        embeddings = remove_anchor_outliers(
            embeddings, num_outliers=drop_anchor_outliers
        )

    if average_anchors:
        embeddings = reduce(embeddings, "b n f -> b f", reduction="mean")

    if drop_batch_outliers > 0:
        embeddings = remove_batch_outliers(embeddings, num_outliers=drop_batch_outliers)

    if average_batch:
        embeddings = reduce(embeddings, "b f -> 1 f", reduction="mean")

    if normalize_embeddings:
        embeddings = F.normalize(embeddings, p=2, dim=-1)

    embeddings = embeddings.view(-1, embeddings.shape[-1])
    return embeddings.detach().numpy()


def get_samples_UNSW(df, speed, load, severity):
    samples = df[
        (df["speed"] == speed) & (df["load"] == load) & (df["severity"] == severity)
    ]
    samples = np.array(list(samples["signal"]))
    samples = samples[
        :,
        get_important_frequencies(
            UNSW_teeth,
            config["gear_harmonics_to_include"],
            config["gear_sideband_block_size"],
        ),
    ]
    samples = np.log10(samples)

    return samples


def get_samples_MCC5(df, speed, load, fault_class, severity, circulation=None):
    samples = df[
        (df["speed"] == speed)
        & (df["load"] == load)
        & (df["severity"] == severity)
        & (df["class"] == fault_class)
    ]

    # If None, take both torque and speed circulations
    if circulation is not None:
        samples = samples[(samples["circulation"] == circulation)]

    samples = np.array(list(samples["signal"]))
    samples = samples[
        :,
        get_important_frequencies(
            MCC5_teeth,
            config["gear_harmonics_to_include"],
            config["gear_sideband_block_size"],
        ),
    ]
    samples = np.log10(samples)

    return samples


def get_samples_AGFD(
    df, speed, load, fault_class, severity, installation=None, healthy_GP=None
):
    samples = df[
        (df["speed"] == speed)
        & (df["load"] == load)
        & (df["severity"] == severity)
        & (df["class"] == fault_class)
    ]

    if installation is not None:
        samples = samples[(samples["installation"] == installation)]

    if healthy_GP is not None:
        samples = samples[(samples["healthy_GP"] == healthy_GP)]

    samples = np.array(list(samples["signal"]))
    samples = samples[
        :,
        get_important_frequencies(
            AGFD_teeth,
            config["gear_harmonics_to_include"],
            config["gear_sideband_block_size"],
        ),
    ]
    samples = np.log10(samples)

    return samples

## Data


### Data UNSW


In [ ]:
# Load data
UNSW_crack_df = pd.read_feather(Path.cwd() / "data" / "UNSW_gear_crack_OT_V2.feather")

# Change S crack to second healthy
UNSW_crack_severity_fix_map = {"H": "H1", "S": "H2"}
UNSW_crack_df["severity"] = UNSW_crack_df["severity"].transform(
    lambda x: UNSW_crack_severity_fix_map.get(x, x)
)

# Need only one OT method
UNSW_crack_df = UNSW_crack_df[UNSW_crack_df["OT_method"] == "signal"]

# Drop unnecessary columns
UNSW_crack_df.drop(
    columns=["OT_method", "TSA_size"],
    inplace=True,
)

UNSW_crack_df[["speed", "load", "severity"]].value_counts().sort_index()

### Data MCC5


In [ ]:
# Load data
MCC5_df = pd.read_feather(Path.cwd() / "data" / "MCC5-THU_OT_V2.feather")

# Need only one OT method
MCC5_df = MCC5_df[MCC5_df["OT_method"] == "signal"]
# Should only be 30 in this file, but just in case
MCC5_df = MCC5_df[MCC5_df["TSA_size"] == 30]

# Drop unnecessary columns
MCC5_df.drop(
    columns=["OT_method", "TSA_size"],
    inplace=True,
)
# Only use gearbox_vibration_x sensor
MCC5_df = MCC5_df[
    ["speed", "load", "severity", "class", "circulation", "gearbox_vibration_x"]
]
MCC5_df.rename(columns={"gearbox_vibration_x": "signal"}, inplace=True)

# Drop in-between speeds and loads
MCC5_df = MCC5_df[MCC5_df["speed"].isin([1000, 2000, 3000])]
MCC5_df = MCC5_df[MCC5_df["load"].isin([10, 20])]

# Keep only crack, wear, and healthy
MCC5_df = MCC5_df[MCC5_df["class"].isin(["crack", "wear", "healthy"])]
MCC5_df["class"].value_counts()

MCC5_df[["speed", "load", "severity", "class"]].value_counts().sort_index()

In [ ]:
MCC5_df[MCC5_df["speed"] == 2500][
    ["speed", "load", "severity", "class", "circulation"]
].value_counts().sort_index()

### Data AGFD


In [ ]:
# Load data
AGFD_df = pd.read_feather(Path.cwd() / "data" / "AGFD_OT_V2.feather")


# Need only one OT method
AGFD_df = AGFD_df[AGFD_df["OT_method"] == "signal"]
# Should only be 40 in this file, but just in case
AGFD_df = AGFD_df[AGFD_df["TSA_size"] == 40]

# # Drop unnecessary columns
AGFD_df.drop(
    columns=["OT_method", "TSA_size", "acc2", "acc3"],
    inplace=True,
)
# Use only Acc4
AGFD_df.rename(columns={"acc4": "signal"}, inplace=True)

AGFD_df = AGFD_df[AGFD_df["speed"] != 250]


AGFD_df[
    ["speed", "load", "class", "severity", "healthy_GP", "installation"]
].value_counts().sort_index()
# AGFD_df.value_counts().sort_index()

In [ ]:
AGFD_df[["speed", "load", "class", "severity", "healthy_GP", "installation"]][
    AGFD_df["class"] == "healthy"
].value_counts().sort_index()

## Results [Ensemble]


### Results on UNSW


In [ ]:
# Embed everything with ensemble

# ensemble_weight_dir_name = "SCENARIO_UNSW"
# ensemble_weight_dir_name = "SCENARIO_UNSW_AGFD_ONLY"
ensemble_weight_dir_name = "SCENARIO_UNSW_MCC5_ONLY"

ensemble_weight_dirs = sorted(
    list(os.walk(model_weight_dir / ensemble_weight_dir_name))[0][1]
)[(ensemble_repeat_num - 1) * 5 : ensemble_repeat_num * 5]
UNSW_ensemble_emb_dfs = []

for e in range(5):
    # Load ensemble models
    model.backbone.load_state_dict(
        torch.load(
            model_weight_dir
            / ensemble_weight_dir_name
            / ensemble_weight_dirs[e]
            / f"{ensemble_checkpoint_num}.pth",
            # model_weight_dir
            # / f"{ensemble_weight_dir_name} R{ensemble_repeat_num}E{e + 1}"
            # / f"{ensemble_checkpoint_num}.pth",
            weights_only=True,
        )
    )
    model.eval()

    # Embeddings for one ensemble model
    all_emb_dfs = []
    for speed in [5, 10, 15, 20]:
        for load in [5, 10, 15]:
            # ANCHOR
            #

            if config["use_difference"]:
                # * Use 16 first samples for anchors. Slowest speed has 32 samples, so it's 50% of that.
                # * 16 from each to have consistent number of anchor samples.
                anchor_samples = get_samples_UNSW(UNSW_crack_df, speed, load, "H1")[:16]

                # print(anchor_samples.shape)
                anchor_samples = anchor_samples.reshape(
                    1,
                    16,
                    anchor_samples.shape[1],
                    # 4, 4, anchor_samples.shape[1]
                ).mean(axis=1)
                # print(anchor_samples.shape)

            # HEALTHY 1
            #

            # Get samples from specific conditions
            # * Drop 16 first from H1, as those are used for anchors
            healthy_samples_1 = get_samples_UNSW(UNSW_crack_df, speed, load, "H1")[16:]
            if config["use_difference"]:
                # Calculate difference against anchor samples
                healthy_diff_1 = (
                    healthy_samples_1[:, np.newaxis, :]
                    - anchor_samples[np.newaxis, :, :]
                )
            else:
                # Just add new axis for anchor dimension
                healthy_diff_1 = healthy_samples_1[:, np.newaxis, :]

            # Create embeddings with pretrained model
            healthy_embeddings_1 = embed(healthy_diff_1, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            healthy_emb_df_1 = pd.DataFrame(
                healthy_embeddings_1, columns=embedding_columns
            )
            healthy_emb_df_1["class"] = "healthy"
            healthy_emb_df_1["severity"] = "H1"

            # HEALTHY 2
            #

            # Get samples from specific conditions
            healthy_samples_2 = get_samples_UNSW(UNSW_crack_df, speed, load, "H2")
            # Calculate difference against anchor samples
            healthy_diff_2 = (
                healthy_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            healthy_embeddings_2 = embed(healthy_diff_2, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            healthy_emb_df_2 = pd.DataFrame(
                healthy_embeddings_2, columns=embedding_columns
            )
            healthy_emb_df_2["class"] = "healthy"
            healthy_emb_df_2["severity"] = "H2"

            # CRACK 1
            #

            # Get samples from specific conditions
            crack_samples_1 = get_samples_UNSW(UNSW_crack_df, speed, load, "M")
            # Calculate difference against anchor samples
            crack_diff_1 = (
                crack_samples_1[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            crack_embeddings_1 = embed(crack_diff_1, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            crack_emb_df_1 = pd.DataFrame(crack_embeddings_1, columns=embedding_columns)
            crack_emb_df_1["class"] = "crack"
            crack_emb_df_1["severity"] = "M"

            # CRACK 2
            #

            # Get samples from specific conditions
            crack_samples_2 = get_samples_UNSW(UNSW_crack_df, speed, load, "L")
            # Calculate difference against anchor samples
            crack_diff_2 = (
                crack_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            crack_embeddings_2 = embed(crack_diff_2, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            crack_emb_df_2 = pd.DataFrame(crack_embeddings_2, columns=embedding_columns)
            crack_emb_df_2["class"] = "crack"
            crack_emb_df_2["severity"] = "L"

            # Combine all dataframes and add back operating condition info
            emb_df = pd.concat(
                [
                    healthy_emb_df_1,
                    healthy_emb_df_2,
                    crack_emb_df_1,
                    crack_emb_df_2,
                ],
                ignore_index=True,
            )
            emb_df["speed"] = speed
            emb_df["load"] = load

            all_emb_dfs.append(emb_df)

    all_emb_dfs = pd.concat(all_emb_dfs, ignore_index=True)
    UNSW_ensemble_emb_dfs.append(all_emb_dfs)

In [ ]:
emb = UNSW_ensemble_emb_dfs[0]
# emb = emb[emb["speed"] == 3000]
# emb = emb[emb["load"] == 10]
# emb = emb[emb["severity"].isin(["H1", "H2", "L"])]

fig = px.scatter(
    emb,
    x="x",
    y="y",
    color="class",
    # symbol="speed",
    symbol="severity",
    # symbol="load",
    hover_data=["speed", "load", "class", "severity"],
    title="UNSW",
)
fig.show()

In [ ]:
# 1 operating condition for prototypes
# n - 1 operating conditions for query

UNSW_accuracies = []
# For every operating conditions (consisting of speed and load)
for speed in [5, 10, 15, 20]:
    for load in [5, 10, 15]:
        # Repeat with every ensemble model
        ensemble_distances_M = []
        ensemble_distances_L = []
        for e in range(5):
            prototypes = UNSW_ensemble_emb_dfs[e][
                (UNSW_ensemble_emb_dfs[e]["speed"] == speed)
                & (UNSW_ensemble_emb_dfs[e]["load"] == load)
            ]

            # Get possible prototypes

            healthy_prototype_H1 = (
                prototypes[prototypes["severity"] == "H1"][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            healthy_prototype_H2 = (
                prototypes[prototypes["severity"] == "H2"][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_M = (
                prototypes[prototypes["severity"] == "M"][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_L = (
                prototypes[prototypes["severity"] == "L"][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            # Use everything else for query
            # (Not H1 [used for anchor] and not samples from operating condition used for prototypes)
            query_df = UNSW_ensemble_emb_dfs[e][
                ~(
                    (
                        (UNSW_ensemble_emb_dfs[e]["speed"] == speed)
                        & (UNSW_ensemble_emb_dfs[e]["load"] == load)
                    )
                    | (UNSW_ensemble_emb_dfs[e]["severity"] == "H1")
                )
            ]

            # Need separate queries from when M or L are used as the faulty prototype
            query_df_M = query_df[query_df["severity"] != "L"]
            query_df_L = query_df[query_df["severity"] != "M"]

            # Convert severities to target labels [0, 1]
            targets_M = query_df_M["severity"].map({"H2": 0, "M": 1, "L": 1}).to_numpy()
            targets_L = query_df_L["severity"].map({"H2": 0, "M": 1, "L": 1}).to_numpy()

            query_embs_M = query_df_M[["x", "y"]].to_numpy()
            query_embs_L = query_df_L[["x", "y"]].to_numpy()

            # Calculate query distances to prototypes
            #   H1 always used as healthy prototype
            #   M & L crack severities used both ways
            distances_M = np.stack(
                [
                    np.linalg.norm(
                        query_embs_M - healthy_prototype_H1[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_M - crack_prototype_L[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            distances_L = np.stack(
                [
                    np.linalg.norm(
                        query_embs_L - healthy_prototype_H1[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_L - crack_prototype_M[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )

            ensemble_distances_M.append(distances_M)
            ensemble_distances_L.append(distances_L)

        # Soft voting
        voted_distances_M = np.mean(np.stack(ensemble_distances_M, axis=-1), axis=-1)
        voted_distances_L = np.mean(np.stack(ensemble_distances_L, axis=-1), axis=-1)

        acc_M = np.mean(np.argmin(voted_distances_M, axis=1) == targets_M)
        acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_M:>10.2%} (M), {acc_L:>10.2%} (L)")
        UNSW_accuracies.append([acc_M, acc_L])

print(f"UNSW Average Accuracy: {np.mean(UNSW_accuracies):.4%}")

In [ ]:
# n - 1 operating conditions for prototypes
# 1 operating condition for query

UNSW_accuracies = []
for speed in [5, 10, 15, 20]:
    for load in [5, 10, 15]:
        ensemble_distances_M = []
        ensemble_distances_L = []
        for e in range(5):
            query_df = UNSW_ensemble_emb_dfs[e][
                (UNSW_ensemble_emb_dfs[e]["speed"] == speed)
                & (UNSW_ensemble_emb_dfs[e]["load"] == load)
                & (UNSW_ensemble_emb_dfs[e]["severity"] != "H1")
            ]

            # Need separate queries from when M or L are used as the faulty prototype
            query_df_M = query_df[query_df["severity"] != "L"]
            query_df_L = query_df[query_df["severity"] != "M"]

            healthy_prototype_H1 = (
                UNSW_ensemble_emb_dfs[e][
                    (UNSW_ensemble_emb_dfs[e]["severity"] == "H1")
                    & (UNSW_ensemble_emb_dfs[e]["speed"] != speed)
                    & (UNSW_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            healthy_prototype_H2 = (
                UNSW_ensemble_emb_dfs[e][
                    (UNSW_ensemble_emb_dfs[e]["severity"] == "H2")
                    & (UNSW_ensemble_emb_dfs[e]["speed"] != speed)
                    & (UNSW_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_M = (
                UNSW_ensemble_emb_dfs[e][
                    (UNSW_ensemble_emb_dfs[e]["severity"] == "M")
                    & (UNSW_ensemble_emb_dfs[e]["speed"] != speed)
                    & (UNSW_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_L = (
                UNSW_ensemble_emb_dfs[e][
                    (UNSW_ensemble_emb_dfs[e]["severity"] == "L")
                    & (UNSW_ensemble_emb_dfs[e]["speed"] != speed)
                    & (UNSW_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            # Convert severities to target labels [0, 1]
            targets_M = (
                query_df_M["severity"]
                .map({"H1": 0, "H2": 0, "M": 1, "L": 1})
                .to_numpy()
            )
            targets_L = (
                query_df_L["severity"]
                .map({"H1": 0, "H2": 0, "M": 1, "L": 1})
                .to_numpy()
            )

            query_embs_M = query_df_M[["x", "y"]].to_numpy()
            query_embs_L = query_df_L[["x", "y"]].to_numpy()

            # query_embs - healthy_prototype_H1
            distances_M = np.stack(
                [
                    np.linalg.norm(
                        query_embs_M - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_M - crack_prototype_L[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            distances_L = np.stack(
                [
                    np.linalg.norm(
                        query_embs_L - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_L - crack_prototype_M[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            ensemble_distances_M.append(distances_M)
            ensemble_distances_L.append(distances_L)

        # Soft voting
        voted_distances_M = np.mean(np.stack(ensemble_distances_M, axis=-1), axis=-1)
        voted_distances_L = np.mean(np.stack(ensemble_distances_L, axis=-1), axis=-1)

        acc_M = np.mean(np.argmin(voted_distances_M, axis=1) == targets_M)
        acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_M:>10.2%} (M), {acc_L:>10.2%} (L)")
        UNSW_accuracies.append((acc_M, acc_L))

print(f"UNSW Average Accuracy: {np.mean(UNSW_accuracies):.4%}")

#### Non-operating cross


In [ ]:
# 1 operating condition for prototypes
# n - 1 operating conditions for query

UNSW_accuracies = []
# For every operating conditions (consisting of speed and load)
for speed in [5, 10, 15, 20]:
    for load in [5, 10, 15]:
        for support_severity, query_severity in [("M", "L"), ("L", "M")]:
            # Repeat with every ensemble model
            ensemble_distances = []
            for e in range(5):
                prototypes = UNSW_ensemble_emb_dfs[e][
                    (UNSW_ensemble_emb_dfs[e]["speed"] == speed)
                    & (UNSW_ensemble_emb_dfs[e]["load"] == load)
                    & (
                        UNSW_ensemble_emb_dfs[e]["severity"].isin([
                            support_severity,
                            "H1",
                        ])
                    )
                ]

                # Get possible prototypes

                healthy_prototype_H1 = (
                    prototypes[prototypes["severity"] == "H1"][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )
                # healthy_prototype_H2 = (
                #     prototypes[prototypes["severity"] == "H2"][["x", "y"]]
                #     .to_numpy()
                #     .mean(axis=0)
                # )
                crack_prototype = (
                    prototypes[prototypes["severity"] == support_severity][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )

                # Use everything else for query
                # (Not H1 [used for anchor] and not samples from operating condition used for prototypes)
                query_df = UNSW_ensemble_emb_dfs[e][
                    (UNSW_ensemble_emb_dfs[e]["speed"] == speed)
                    & (UNSW_ensemble_emb_dfs[e]["load"] == load)
                    & (
                        UNSW_ensemble_emb_dfs[e]["severity"].isin([
                            query_severity,
                            "H2",
                        ])
                    )
                ]

                # Convert severities to target labels [0, 1]
                targets = query_df["severity"].map({"H2": 0, "M": 1, "L": 1}).to_numpy()

                query_embs = query_df[["x", "y"]].to_numpy()

                # Calculate query distances to prototypes
                #   H1 always used as healthy prototype
                #   M & L crack severities used both ways
                distances = np.stack(
                    [
                        np.linalg.norm(
                            query_embs - healthy_prototype_H1[np.newaxis, :],
                            axis=1,
                        ),
                        np.linalg.norm(
                            query_embs - crack_prototype[np.newaxis, :], axis=1
                        ),
                    ],
                    axis=1,
                )

                ensemble_distances.append(distances)

            # Soft voting
            voted_distances = np.mean(np.stack(ensemble_distances, axis=-1), axis=-1)

            acc = np.mean(np.argmin(voted_distances, axis=1) == targets)
            print(
                f"{speed:<2} Hz, {load:<2} Nm: {acc:>10.2%} ({support_severity} -> {query_severity})"
            )
            UNSW_accuracies.append([acc])

print(f"UNSW Average Accuracy: {np.mean(UNSW_accuracies):.4%}")

### Results on MCC5


In [ ]:
# Embed everything with ensemble

# ensemble_weight_dir_name = "SCENARIO_UNSW_MCC5_ONLY"  # XXX
# ensemble_weight_dir_name = "SCENARIO_UNSW_AGFD_ONLY"  # XXX
# ensemble_weight_dir_name = "EXPERIMENTAL"  # XXX
ensemble_weight_dir_name = "SCENARIO_UNSW_UNSW_ONLY"  # XXX
# ensemble_weight_dir_name = "SCENARIO_UNSW_UM"  # XXX
# ensemble_weight_dir_name = "SCENARIO_UNSW_M_5000"  # XXX
# ensemble_weight_dir_name = "SCENARIO_UNSW_UA"  # Trained on UNSW & AGFD
# ensemble_weight_dir_name = "SCENARIO_UNSW_U" # Trained on UNSW
# ensemble_weight_dir_name = "SCENARIO_UNSW_A" # Trained on AGFD

ensemble_weight_dirs = sorted(
    list(os.walk(model_weight_dir / ensemble_weight_dir_name))[0][1]
)[(ensemble_repeat_num - 1) * 5 : ensemble_repeat_num * 5]
MCC5_ensemble_emb_dfs = []

for e in range(5):
    # Load ensemble models
    model.backbone.load_state_dict(
        torch.load(
            model_weight_dir
            / ensemble_weight_dir_name
            / ensemble_weight_dirs[e]
            / f"{ensemble_checkpoint_num}.pth",
            weights_only=True,
        )
    )
    model.eval()

    # Embeddings for one ensemble model
    all_emb_dfs = []
    for speed in [1000, 2000, 3000]:
        for load in [10, 20]:
            # ANCHOR
            #

            if config["use_difference"]:
                # * Use 7 first samples for anchors. Slowest speed has 14 samples, so it's 50% of that.
                # * 7 from each to have consistent number of anchor samples.
                anchor_samples = get_samples_MCC5(
                    MCC5_df, speed, load, "healthy", "-", circulation="torque"
                )[:7]

                # print(anchor_samples.shape)
                anchor_samples = anchor_samples.reshape(
                    1,
                    7,
                    anchor_samples.shape[1],
                    # 4, 4, anchor_samples.shape[1]
                ).mean(axis=1)
                # print(anchor_samples.shape)

            # HEALTHY 1
            #

            # Get samples from specific conditions
            # * Drop 7 first from H1, as those are used for anchors
            healthy_samples_1 = get_samples_MCC5(
                MCC5_df, speed, load, "healthy", "-", circulation="torque"
            )[7:]

            if config["use_difference"]:
                # Calculate difference against anchor samples
                healthy_diff_1 = (
                    healthy_samples_1[:, np.newaxis, :]
                    - anchor_samples[np.newaxis, :, :]
                )
            else:
                # Just add new axis for anchor dimension
                healthy_diff_1 = healthy_samples_1[:, np.newaxis, :]

            # Create embeddings with pretrained model
            healthy_embeddings_1 = embed(healthy_diff_1, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            healthy_emb_df_1 = pd.DataFrame(
                healthy_embeddings_1, columns=embedding_columns
            )
            healthy_emb_df_1["class"] = "healthy"
            healthy_emb_df_1["severity"] = "H1"

            # HEALTHY 2
            #

            # Get samples from specific conditions
            healthy_samples_2 = get_samples_MCC5(
                MCC5_df, speed, load, "healthy", "-", circulation="speed"
            )  # [7:]
            # Calculate difference against anchor samples
            healthy_diff_2 = (
                healthy_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            healthy_embeddings_2 = embed(healthy_diff_2, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            healthy_emb_df_2 = pd.DataFrame(
                healthy_embeddings_2, columns=embedding_columns
            )
            healthy_emb_df_2["class"] = "healthy"
            healthy_emb_df_2["severity"] = "H2"

            # CRACK 1
            #

            # Get samples from specific conditions
            crack_samples_1 = get_samples_MCC5(MCC5_df, speed, load, "crack", "M")
            # Calculate difference against anchor samples
            crack_diff_1 = (
                crack_samples_1[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            crack_embeddings_1 = embed(crack_diff_1, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            crack_emb_df_1 = pd.DataFrame(crack_embeddings_1, columns=embedding_columns)
            crack_emb_df_1["class"] = "crack"
            crack_emb_df_1["severity"] = "M"

            # CRACK 2
            #

            # Get samples from specific conditions
            crack_samples_2 = get_samples_MCC5(MCC5_df, speed, load, "crack", "L")
            # Calculate difference against anchor samples
            crack_diff_2 = (
                crack_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            crack_embeddings_2 = embed(crack_diff_2, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            crack_emb_df_2 = pd.DataFrame(crack_embeddings_2, columns=embedding_columns)
            crack_emb_df_2["class"] = "crack"
            crack_emb_df_2["severity"] = "L"

            # WEAR 1
            #

            # Get samples from specific conditions
            wear_samples_1 = get_samples_MCC5(MCC5_df, speed, load, "wear", "M")
            # Calculate difference against anchor samples
            wear_diff_1 = (
                wear_samples_1[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            wear_embeddings_1 = embed(wear_diff_1, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            wear_emb_df_1 = pd.DataFrame(wear_embeddings_1, columns=embedding_columns)
            wear_emb_df_1["class"] = "wear"
            wear_emb_df_1["severity"] = "M"

            # WEAR 2
            #

            # Get samples from specific conditions
            wear_samples_2 = get_samples_MCC5(MCC5_df, speed, load, "wear", "L")
            # Calculate difference against anchor samples
            wear_diff_2 = (
                wear_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            wear_embeddings_2 = embed(wear_diff_2, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            wear_emb_df_2 = pd.DataFrame(wear_embeddings_2, columns=embedding_columns)
            wear_emb_df_2["class"] = "wear"
            wear_emb_df_2["severity"] = "L"

            # Combine all dataframes and add back operating condition info
            emb_df = pd.concat(
                [
                    healthy_emb_df_1,
                    healthy_emb_df_2,
                    crack_emb_df_1,
                    crack_emb_df_2,
                    wear_emb_df_1,
                    wear_emb_df_2,
                ],
                ignore_index=True,
            )
            emb_df["speed"] = speed
            emb_df["load"] = load

            all_emb_dfs.append(emb_df)

    all_emb_dfs = pd.concat(all_emb_dfs, ignore_index=True)
    MCC5_ensemble_emb_dfs.append(all_emb_dfs)

In [ ]:
emb = MCC5_ensemble_emb_dfs[0]
# emb = emb[emb["speed"] == 3000]
# emb = emb[emb["load"] == 10]
emb = emb[emb["severity"].isin(["H1", "H2", "L"])]

fig = px.scatter(
    emb,
    x="x",
    y="y",
    color="class",
    # symbol="speed",
    # symbol="severity",
    symbol="load",
    hover_data=["class", "speed", "load", "severity"],
    title="MCC5",
)
fig.show()

In [ ]:
# 1 operating condition for prototypes
# n - 1 operating conditions for query

MCC5_accuracies = []
# For every operating conditions (consisting of speed and load)
for speed in [1000, 2000, 3000]:
    # for load in [20]:
    for load in [10, 20]:
        # Repeat with every ensemble model
        ensemble_distances_M = []
        ensemble_distances_L = []
        for e in range(5):
            prototypes = MCC5_ensemble_emb_dfs[e][
                (MCC5_ensemble_emb_dfs[e]["speed"] == speed)
                & (MCC5_ensemble_emb_dfs[e]["load"] == load)
            ]

            # Get possible prototypes

            healthy_prototype_H1 = (
                prototypes[
                    (prototypes["class"] == "healthy")
                    & (prototypes["severity"] == "H1")
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            # healthy_prototype_H2 = (
            #     prototypes[prototypes["severity"] == "H2"][["x", "y"]]
            #     .to_numpy()
            #     .mean(axis=0)
            # )

            crack_prototype_M = (
                prototypes[
                    (prototypes["class"] == "crack") & (prototypes["severity"] == "M")
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_L = (
                prototypes[
                    (prototypes["class"] == "crack") & (prototypes["severity"] == "L")
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            wear_prototype_M = (
                prototypes[
                    (prototypes["class"] == "wear") & (prototypes["severity"] == "M")
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            wear_prototype_L = (
                prototypes[
                    (prototypes["class"] == "wear") & (prototypes["severity"] == "L")
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            # Use everything else for query
            # (Not H1 [used for anchor] and not samples from operating condition used for prototypes)
            query_df = MCC5_ensemble_emb_dfs[e][
                ~(
                    (
                        (MCC5_ensemble_emb_dfs[e]["speed"] == speed)
                        & (MCC5_ensemble_emb_dfs[e]["load"] == load)
                    )
                    | (MCC5_ensemble_emb_dfs[e]["severity"] == "H1")
                    | (MCC5_ensemble_emb_dfs[e]["class"] == "wear")  # XXX
                )
            ]

            # Need separate queries from when M or L are used as the faulty prototype
            query_df_M = query_df[query_df["severity"] != "L"]
            query_df_L = query_df[query_df["severity"] != "M"]

            # Convert severities to target labels [0, 1, 2]
            targets_M = (
                query_df_M["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )
            targets_L = (
                query_df_L["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )

            query_embs_M = query_df_M[["x", "y"]].to_numpy()
            query_embs_L = query_df_L[["x", "y"]].to_numpy()

            # Calculate query distances to prototypes
            #   H1 always used as healthy prototype
            #   S & L crack severities used both ways
            distances_M = np.stack(
                [
                    np.linalg.norm(
                        query_embs_M - healthy_prototype_H1[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_M - crack_prototype_L[np.newaxis, :], axis=1
                    ),
                    np.linalg.norm(
                        query_embs_M - wear_prototype_L[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            distances_L = np.stack(
                [
                    np.linalg.norm(
                        query_embs_L - healthy_prototype_H1[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_L - crack_prototype_M[np.newaxis, :], axis=1
                    ),
                    np.linalg.norm(
                        query_embs_L - wear_prototype_M[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )

            ensemble_distances_M.append(distances_M)
            ensemble_distances_L.append(distances_L)

        # Soft voting
        voted_distances_M = np.mean(np.stack(ensemble_distances_M, axis=-1), axis=-1)
        voted_distances_L = np.mean(np.stack(ensemble_distances_L, axis=-1), axis=-1)

        acc_M = np.mean(np.argmin(voted_distances_M, axis=1) == targets_M)
        acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_M:>10.2%} (M), {acc_L:>10.2%} (L)")
        MCC5_accuracies.append([acc_M, acc_L])

print(f"MCC5 average accuracy: {np.mean(MCC5_accuracies):.4%}")

In [ ]:
# n - 1 operating conditions for prototypes
# 1 operating condition for query

MCC5_accuracies = []
for speed in [1000, 2000, 3000]:
    for load in [10, 20]:
        ensemble_distances_M = []
        ensemble_distances_L = []
        # FIXME
        for e in range(5):
            query_df = MCC5_ensemble_emb_dfs[e][
                (MCC5_ensemble_emb_dfs[e]["speed"] == speed)
                & (MCC5_ensemble_emb_dfs[e]["load"] == load)
                & (MCC5_ensemble_emb_dfs[e]["severity"] != "H1")
            ]

            # Need separate queries from when M or L are used as the faulty prototype
            query_df_M = query_df[query_df["severity"] != "L"]
            query_df_L = query_df[query_df["severity"] != "M"]

            healthy_prototype_H1 = (
                MCC5_ensemble_emb_dfs[e][
                    (MCC5_ensemble_emb_dfs[e]["class"] == "healthy")
                    & (MCC5_ensemble_emb_dfs[e]["severity"] == "H1")
                    & (MCC5_ensemble_emb_dfs[e]["speed"] != speed)
                    & (MCC5_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            healthy_prototype_H2 = (
                MCC5_ensemble_emb_dfs[e][
                    (MCC5_ensemble_emb_dfs[e]["class"] == "healthy")
                    & (MCC5_ensemble_emb_dfs[e]["severity"] == "H2")
                    & (MCC5_ensemble_emb_dfs[e]["speed"] != speed)
                    & (MCC5_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            crack_prototype_M = (
                MCC5_ensemble_emb_dfs[e][
                    (MCC5_ensemble_emb_dfs[e]["class"] == "crack")
                    & (MCC5_ensemble_emb_dfs[e]["severity"] == "M")
                    & (MCC5_ensemble_emb_dfs[e]["speed"] != speed)
                    & (MCC5_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_L = (
                MCC5_ensemble_emb_dfs[e][
                    (MCC5_ensemble_emb_dfs[e]["class"] == "crack")
                    & (MCC5_ensemble_emb_dfs[e]["severity"] == "L")
                    & (MCC5_ensemble_emb_dfs[e]["speed"] != speed)
                    & (MCC5_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            wear_prototype_M = (
                MCC5_ensemble_emb_dfs[e][
                    (MCC5_ensemble_emb_dfs[e]["class"] == "wear")
                    & (MCC5_ensemble_emb_dfs[e]["severity"] == "M")
                    & (MCC5_ensemble_emb_dfs[e]["speed"] != speed)
                    & (MCC5_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            wear_prototype_L = (
                MCC5_ensemble_emb_dfs[e][
                    (MCC5_ensemble_emb_dfs[e]["class"] == "wear")
                    & (MCC5_ensemble_emb_dfs[e]["severity"] == "L")
                    & (MCC5_ensemble_emb_dfs[e]["speed"] != speed)
                    & (MCC5_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            # Convert severities to target labels [0, 1, 2]
            targets_M = (
                query_df_M["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )
            targets_L = (
                query_df_L["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )

            query_embs_M = query_df_M[["x", "y"]].to_numpy()
            query_embs_L = query_df_L[["x", "y"]].to_numpy()

            # query_embs - healthy_prototype_H1
            distances_M = np.stack(
                [
                    np.linalg.norm(
                        query_embs_M - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_M - crack_prototype_L[np.newaxis, :], axis=1
                    ),
                    np.linalg.norm(
                        query_embs_M - wear_prototype_L[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            distances_L = np.stack(
                [
                    np.linalg.norm(
                        query_embs_L - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_L - crack_prototype_M[np.newaxis, :], axis=1
                    ),
                    np.linalg.norm(
                        query_embs_L - wear_prototype_M[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            ensemble_distances_M.append(distances_M)
            ensemble_distances_L.append(distances_L)

        # Soft voting
        voted_distances_M = np.mean(np.stack(ensemble_distances_M, axis=-1), axis=-1)
        voted_distances_L = np.mean(np.stack(ensemble_distances_L, axis=-1), axis=-1)

        acc_M = np.mean(np.argmin(voted_distances_M, axis=1) == targets_M)
        acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_M:>10.2%} (M), {acc_L:>10.2%} (L)")
        MCC5_accuracies.append((acc_M, acc_L))

print(f"MCC5 average accuracy: {np.mean(MCC5_accuracies):.4%}")

In [ ]:
# 1 severity for prototypes
# 1 severity for query

severity = "L"

MCC5_accuracies = []
for speed in [1000, 2000, 3000]:
    for load in [10, 20]:
        for support_severity, query_severity in [["M", "L"], ["L", "M"]]:
            ensemble_distances = []
            for e in range(5):
                query_df = MCC5_ensemble_emb_dfs[e][
                    (MCC5_ensemble_emb_dfs[e]["speed"] == speed)
                    & (MCC5_ensemble_emb_dfs[e]["load"] == load)
                    & (
                        MCC5_ensemble_emb_dfs[e]["severity"].isin([
                            "H1",
                            query_severity,
                        ])
                    )
                ]

                healthy_prototype_H1 = (
                    MCC5_ensemble_emb_dfs[e][
                        (MCC5_ensemble_emb_dfs[e]["class"] == "healthy")
                        & (MCC5_ensemble_emb_dfs[e]["severity"] == "H1")
                        & (MCC5_ensemble_emb_dfs[e]["speed"] == speed)
                        & (MCC5_ensemble_emb_dfs[e]["load"] == load)
                    ][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )
                # healthy_prototype_H2 = (
                #     MCC5_ensemble_emb_dfs[e][
                #         (MCC5_ensemble_emb_dfs[e]["class"] == "healthy")
                #         & (MCC5_ensemble_emb_dfs[e]["severity"] == "H2")
                #         & (MCC5_ensemble_emb_dfs[e]["speed"] == speed)
                #         & (MCC5_ensemble_emb_dfs[e]["load"] == load)
                #     ][["x", "y"]]
                #     .to_numpy()
                #     .mean(axis=0)
                # )

                crack_prototype = (
                    MCC5_ensemble_emb_dfs[e][
                        (MCC5_ensemble_emb_dfs[e]["class"] == "crack")
                        & (MCC5_ensemble_emb_dfs[e]["severity"] == support_severity)
                        & (MCC5_ensemble_emb_dfs[e]["speed"] == speed)
                        & (MCC5_ensemble_emb_dfs[e]["load"] == load)
                    ][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )

                wear_prototype = (
                    MCC5_ensemble_emb_dfs[e][
                        (MCC5_ensemble_emb_dfs[e]["class"] == "wear")
                        & (MCC5_ensemble_emb_dfs[e]["severity"] == support_severity)
                        & (MCC5_ensemble_emb_dfs[e]["speed"] != speed)
                        & (MCC5_ensemble_emb_dfs[e]["load"] != load)
                    ][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )

                # Convert severities to target labels [0, 1, 2]
                targets = (
                    query_df["class"]
                    .map({"healthy": 0, "crack": 1, "wear": 2})
                    .to_numpy()
                )
                query_embs = query_df[["x", "y"]].to_numpy()

                # query_embs - healthy_prototype_H1
                distances = np.stack(
                    [
                        np.linalg.norm(
                            query_embs - healthy_prototype_H1[np.newaxis, :],
                            # query_embs - healthy_prototype_H2[np.newaxis, :],
                            axis=1,
                        ),
                        np.linalg.norm(
                            query_embs - crack_prototype[np.newaxis, :], axis=1
                        ),
                        np.linalg.norm(
                            query_embs - wear_prototype[np.newaxis, :], axis=1
                        ),
                    ],
                    axis=1,
                )
                ensemble_distances.append(distances)

            # Soft voting
            voted_distances = np.mean(np.stack(ensemble_distances, axis=-1), axis=-1)

            acc = np.mean(np.argmin(voted_distances, axis=1) == targets)
            print(
                f"{speed:<2} Hz, {load:<2} Nm: {acc:>10.2%} ({support_severity}->{query_severity})"
            )
            MCC5_accuracies.append(acc)

print(f"MCC5 average accuracy: {np.mean(MCC5_accuracies):.4%}")

In [ ]:
# emb = AGFD_ensemble_emb_dfs[0]
emb = query_df

# emb = emb[emb["speed"] == 1000]
# emb = emb[emb["load"] == 6]
# emb = emb[emb["installation"].isin([0, 1, 2])]
emb["role"] = 1

healthy_proto = pd.DataFrame([healthy_prototype_H1], columns=["x", "y"])
healthy_proto["class"] = "healthy"
healthy_proto["severity"] = "H1"
healthy_proto["role"] = 15

crack_proto = pd.DataFrame([crack_prototype], columns=["x", "y"])
crack_proto["class"] = "crack"
crack_proto["severity"] = support_severity
crack_proto["role"] = 15

wear_proto = pd.DataFrame([wear_prototype], columns=["x", "y"])
wear_proto["class"] = "wear"
wear_proto["severity"] = support_severity
wear_proto["role"] = 15

emb = pd.concat([emb, healthy_proto, crack_proto, wear_proto], ignore_index=True)

fig = px.scatter(
    emb,
    x="x",
    y="y",
    color="class",
    # symbol="speed",
    # symbol="load",
    # symbol="severity",
    # symbol="installation",
    size="role",
    hover_data=["speed", "load", "class", "severity"],
    # hover_data=["installation"],
    title="MCC5",
)
fig.show()

### Results on AGFD


In [ ]:
# Embed everything with ensemble

# ensemble_weight_dir_name = "SCENARIO_UNSW_UM"
# ensemble_weight_dir_name = "SCENARIO_UNSW_UNSW_ONLY" # XXX
ensemble_weight_dir_name = "SCENARIO_UNSW_MCC5_ONLY"  # XXX

ensemble_weight_dirs = sorted(
    list(os.walk(model_weight_dir / ensemble_weight_dir_name))[0][1]
)[(ensemble_repeat_num - 1) * 5 : ensemble_repeat_num * 5]
AGFD_ensemble_emb_dfs = []

for e in range(5):
    # Load ensemble models
    model.backbone.load_state_dict(
        torch.load(
            model_weight_dir
            / ensemble_weight_dir_name
            / ensemble_weight_dirs[e]
            / f"{ensemble_checkpoint_num}.pth",
            weights_only=True,
        )
    )
    model.eval()

    # Embeddings for one ensemble model
    all_emb_dfs = []
    for speed in [500, 750, 1000, 1250, 1500]:
        for load in [1, 6, 11]:
            # ANCHOR
            #

            if config["use_difference"]:
                # * 16 should be enough
                anchor_samples = get_samples_AGFD(
                    AGFD_df, speed, load, "healthy", "-", healthy_GP=4
                )[:16]

                # print(anchor_samples.shape)
                anchor_samples = anchor_samples.reshape(
                    1,
                    16,
                    anchor_samples.shape[1],
                    # 4, 4, anchor_samples.shape[1]
                ).mean(axis=1)
                # print(anchor_samples.shape)

            # HEALTHY 1
            #

            # Get samples from specific conditions
            # * Drop 7 first from H1, as those are used for anchors
            healthy_samples_1 = get_samples_AGFD(
                AGFD_df, speed, load, "healthy", "-", healthy_GP=5
            )[16:]

            if config["use_difference"]:
                # Calculate difference against anchor samples
                healthy_diff_1 = (
                    healthy_samples_1[:, np.newaxis, :]
                    - anchor_samples[np.newaxis, :, :]
                )
            else:
                # Just add new axis for anchor dimension
                healthy_diff_1 = healthy_samples_1[:, np.newaxis, :]

            # Create embeddings with pretrained model
            healthy_embeddings_1 = embed(healthy_diff_1, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            healthy_emb_df_1 = pd.DataFrame(
                healthy_embeddings_1, columns=embedding_columns
            )
            healthy_emb_df_1["class"] = "healthy"
            healthy_emb_df_1["severity"] = "H1"
            healthy_emb_df_1["installation"] = 0

            # HEALTHY 2
            #

            # Get samples from specific conditions
            healthy_samples_2 = get_samples_AGFD(
                AGFD_df, speed, load, "healthy", "-", healthy_GP=9
            )
            # Calculate difference against anchor samples
            healthy_diff_2 = (
                healthy_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
            )
            # Create embeddings with pretrained model
            healthy_embeddings_2 = embed(healthy_diff_2, model, **embedding_specs)

            # Convert to dataframe and add other necessary info back (class)
            healthy_emb_df_2 = pd.DataFrame(
                healthy_embeddings_2, columns=embedding_columns
            )
            healthy_emb_df_2["class"] = "healthy"
            healthy_emb_df_2["severity"] = "H2"
            healthy_emb_df_2["installation"] = 0

            crack_emb_dfs_1 = []
            crack_emb_dfs_2 = []
            wear_emb_dfs_1 = []
            wear_emb_dfs_2 = []
            for installation in [1, 2, 3]:
                # CRACK 1
                #

                # Get samples from specific conditions
                crack_samples_1 = get_samples_AGFD(
                    AGFD_df, speed, load, "crack", "S", installation=installation
                )
                # Calculate difference against anchor samples
                crack_diff_1 = (
                    crack_samples_1[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
                )
                # Create embeddings with pretrained model
                crack_embeddings_1 = embed(crack_diff_1, model, **embedding_specs)

                # Convert to dataframe and add other necessary info back (class)
                crack_emb_df_1 = pd.DataFrame(
                    crack_embeddings_1,
                    columns=embedding_columns,
                )
                crack_emb_df_1["class"] = "crack"
                crack_emb_df_1["severity"] = "S"
                crack_emb_df_1["installation"] = installation
                crack_emb_dfs_1.append(crack_emb_df_1)

                # CRACK 2
                #

                # Get samples from specific conditions
                crack_samples_2 = get_samples_AGFD(
                    AGFD_df, speed, load, "crack", "L", installation=installation
                )
                # Calculate difference against anchor samples
                crack_diff_2 = (
                    crack_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
                )
                # Create embeddings with pretrained model
                crack_embeddings_2 = embed(crack_diff_2, model, **embedding_specs)

                # Convert to dataframe and add other necessary info back (class)
                crack_emb_df_2 = pd.DataFrame(
                    crack_embeddings_2, columns=embedding_columns
                )
                crack_emb_df_2["class"] = "crack"
                crack_emb_df_2["severity"] = "L"
                crack_emb_df_2["installation"] = installation
                crack_emb_dfs_2.append(crack_emb_df_2)

                # WEAR 1
                #

                # Get samples from specific conditions
                wear_samples_1 = get_samples_AGFD(
                    AGFD_df, speed, load, "wear", "S", installation=installation
                )
                # Calculate difference against anchor samples
                wear_diff_1 = (
                    wear_samples_1[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
                )
                # Create embeddings with pretrained model
                wear_embeddings_1 = embed(wear_diff_1, model, **embedding_specs)

                # Convert to dataframe and add other necessary info back (class)
                wear_emb_df_1 = pd.DataFrame(
                    wear_embeddings_1, columns=embedding_columns
                )
                wear_emb_df_1["class"] = "wear"
                wear_emb_df_1["severity"] = "S"
                wear_emb_df_1["installation"] = installation
                wear_emb_dfs_1.append(wear_emb_df_1)

                # WEAR 2
                #

                # Get samples from specific conditions
                wear_samples_2 = get_samples_AGFD(
                    AGFD_df, speed, load, "wear", "L", installation=installation
                )
                # Calculate difference against anchor samples
                wear_diff_2 = (
                    wear_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
                )
                # Create embeddings with pretrained model
                wear_embeddings_2 = embed(wear_diff_2, model, **embedding_specs)

                # Convert to dataframe and add other necessary info back (class)
                wear_emb_df_2 = pd.DataFrame(
                    wear_embeddings_2, columns=embedding_columns
                )
                wear_emb_df_2["class"] = "wear"
                wear_emb_df_2["severity"] = "L"
                wear_emb_df_2["installation"] = installation
                wear_emb_dfs_2.append(wear_emb_df_2)

            crack_emb_df_1 = pd.concat(crack_emb_dfs_1, ignore_index=True)
            crack_emb_df_2 = pd.concat(crack_emb_dfs_2, ignore_index=True)
            wear_emb_df_1 = pd.concat(wear_emb_dfs_1, ignore_index=True)
            wear_emb_df_2 = pd.concat(wear_emb_dfs_2, ignore_index=True)

            # Combine all dataframes and add back operating condition info
            emb_df = pd.concat(
                [
                    healthy_emb_df_1,
                    healthy_emb_df_2,
                    crack_emb_df_1,
                    crack_emb_df_2,
                    wear_emb_df_1,
                    wear_emb_df_2,
                ],
                ignore_index=True,
            )
            emb_df["speed"] = speed
            emb_df["load"] = load

            all_emb_dfs.append(emb_df)

    all_emb_dfs = pd.concat(all_emb_dfs, ignore_index=True)
    AGFD_ensemble_emb_dfs.append(all_emb_dfs)

In [ ]:
emb = AGFD_ensemble_emb_dfs[0]
emb = emb[emb["speed"] == 1000]
emb = emb[emb["load"] == 6]
# emb = emb[emb["severity"].isin(["H1", "H2", "L"])]

fig = px.scatter(
    emb,
    x="x",
    y="y",
    color="class",
    # symbol="speed",
    # symbol="load",
    # symbol="severity",
    symbol="installation",
    hover_data=["speed", "load", "class", "severity", "installation"],
    title="AGFD",
)
fig.show()

In [ ]:
# n - 1 operating conditions for prototypes
# 1 operating condition for query

AGFD_accuracies = []
for speed in [500, 750, 1000, 1250, 1500]:
    for load in [1, 6, 11]:
        ensemble_distances_S = []
        ensemble_distances_L = []
        for e in range(5):
            query_df = AGFD_ensemble_emb_dfs[e][
                (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                & (AGFD_ensemble_emb_dfs[e]["severity"] != "H1")
            ]

            # Need separate queries from when M or L are used as the faulty prototype
            query_df_S = query_df[query_df["severity"] != "L"]
            query_df_L = query_df[query_df["severity"] != "S"]

            healthy_prototype_H1 = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "H1")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            healthy_prototype_H2 = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "H2")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            crack_prototype_S = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_L = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            # wear_prototype_S = (
            #     AGFD_ensemble_emb_dfs[e][
            #         (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
            #         & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
            #         & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
            #         & (AGFD_ensemble_emb_dfs[e]["load"] != load)
            #     ][["x", "y"]]
            #     .to_numpy()
            #     .mean(axis=0)
            # )
            # wear_prototype_L = (
            #     AGFD_ensemble_emb_dfs[e][
            #         (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
            #         & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
            #         & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
            #         & (AGFD_ensemble_emb_dfs[e]["load"] != load)
            #     ][["x", "y"]]
            #     .to_numpy()
            #     .mean(axis=0)
            # )

            # Convert severities to target labels [0, 1, 2]
            targets_S = (
                query_df_S["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )
            targets_L = (
                query_df_L["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )

            query_embs_S = query_df_S[["x", "y"]].to_numpy()
            query_embs_L = query_df_L[["x", "y"]].to_numpy()

            # query_embs - healthy_prototype_H1
            distances_S = np.stack(
                [
                    np.linalg.norm(
                        query_embs_S - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_S - crack_prototype_L[np.newaxis, :], axis=1
                    ),
                    # np.linalg.norm(
                    #     query_embs_S - wear_prototype_L[np.newaxis, :], axis=1
                    # ),
                ],
                axis=1,
            )
            distances_L = np.stack(
                [
                    np.linalg.norm(
                        query_embs_L - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_L - crack_prototype_S[np.newaxis, :], axis=1
                    ),
                    # np.linalg.norm(
                    #     query_embs_L - wear_prototype_S[np.newaxis, :], axis=1
                    # ),
                ],
                axis=1,
            )
            ensemble_distances_S.append(distances_S)
            ensemble_distances_L.append(distances_L)

        # Soft voting
        voted_distances_S = np.mean(np.stack(ensemble_distances_S, axis=-1), axis=-1)
        voted_distances_L = np.mean(np.stack(ensemble_distances_L, axis=-1), axis=-1)

        acc_S = np.mean(np.argmin(voted_distances_S, axis=1) == targets_S)
        acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_S:>10.2%} (S), {acc_L:>10.2%} (L)")
        AGFD_accuracies.append((acc_S, acc_L))

print(f"MCC5 average accuracy: {np.mean(AGFD_accuracies):.4%}")

In [ ]:
# n - 1 operating conditions for prototypes
# 1 operating condition for query

AGFD_accuracies = []
for speed in [500, 750, 1000, 1250, 1500]:
    for load in [1, 6, 11]:
        ensemble_distances_S = []
        ensemble_distances_L = []
        # FIXME
        for e in range(5):
            query_df = AGFD_ensemble_emb_dfs[e][
                (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                & (AGFD_ensemble_emb_dfs[e]["severity"] != "H1")
            ]

            # Need separate queries from when M or L are used as the faulty prototype
            query_df_S = query_df[query_df["severity"] != "L"]
            query_df_L = query_df[query_df["severity"] != "S"]

            healthy_prototype_H1 = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "H1")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            healthy_prototype_H2 = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "H2")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            crack_prototype_S = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            crack_prototype_L = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            wear_prototype_S = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )
            wear_prototype_L = (
                AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
                    & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
                    & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                ][["x", "y"]]
                .to_numpy()
                .mean(axis=0)
            )

            # Convert severities to target labels [0, 1, 2]
            targets_S = (
                query_df_S["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )
            targets_L = (
                query_df_L["class"]
                .map({"healthy": 0, "crack": 1, "wear": 2})
                .to_numpy()
            )

            query_embs_S = query_df_S[["x", "y"]].to_numpy()
            query_embs_L = query_df_L[["x", "y"]].to_numpy()

            # query_embs - healthy_prototype_H1
            distances_S = np.stack(
                [
                    np.linalg.norm(
                        query_embs_S - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_S - crack_prototype_L[np.newaxis, :], axis=1
                    ),
                    np.linalg.norm(
                        query_embs_S - wear_prototype_L[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            distances_L = np.stack(
                [
                    np.linalg.norm(
                        query_embs_L - healthy_prototype_H1[np.newaxis, :],
                        # query_embs - healthy_prototype_H2[np.newaxis, :],
                        axis=1,
                    ),
                    np.linalg.norm(
                        query_embs_L - crack_prototype_S[np.newaxis, :], axis=1
                    ),
                    np.linalg.norm(
                        query_embs_L - wear_prototype_S[np.newaxis, :], axis=1
                    ),
                ],
                axis=1,
            )
            ensemble_distances_S.append(distances_S)
            ensemble_distances_L.append(distances_L)

        # Soft voting
        voted_distances_S = np.mean(np.stack(ensemble_distances_S, axis=-1), axis=-1)
        voted_distances_L = np.mean(np.stack(ensemble_distances_L, axis=-1), axis=-1)

        acc_S = np.mean(np.argmin(voted_distances_S, axis=1) == targets_S)
        acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_S:>10.2%} (S), {acc_L:>10.2%} (L)")
        AGFD_accuracies.append((acc_S, acc_L))

print(f"MCC5 average accuracy: {np.mean(AGFD_accuracies):.4%}")

In [ ]:
# n - 1 installations for prototypes
# 1 installation for query

AGFD_accuracies = []
for speed in [500, 750, 1000, 1250, 1500]:
    for load in [1, 6, 11]:
        for installation in [1, 2, 3]:
            ensemble_distances_S = []
            ensemble_distances_L = []
            for e in range(5):
                query_df = AGFD_ensemble_emb_dfs[e][
                    (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                    & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                    & (AGFD_ensemble_emb_dfs[e]["installation"].isin([0, installation]))
                    & (AGFD_ensemble_emb_dfs[e]["severity"] != "H1")
                ]

                # Need separate queries from when M or L are used as the faulty prototype
                query_df_S = query_df[query_df["severity"] != "L"]
                query_df_L = query_df[query_df["severity"] != "S"]

                healthy_prototype_H1 = (
                    AGFD_ensemble_emb_dfs[e][
                        (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                        & (AGFD_ensemble_emb_dfs[e]["severity"] == "H1")
                        & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                        & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                    ][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )
                healthy_prototype_H2 = (
                    AGFD_ensemble_emb_dfs[e][
                        (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                        & (AGFD_ensemble_emb_dfs[e]["severity"] == "H2")
                        & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                        & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                    ][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )

                crack_prototype_S = (
                    AGFD_ensemble_emb_dfs[e][
                        (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
                        & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
                        & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                        & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                        & (AGFD_ensemble_emb_dfs[e]["installation"] != installation)
                    ][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )
                crack_prototype_L = (
                    AGFD_ensemble_emb_dfs[e][
                        (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
                        & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
                        & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                        & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                        & (AGFD_ensemble_emb_dfs[e]["installation"] != installation)
                    ][["x", "y"]]
                    .to_numpy()
                    .mean(axis=0)
                )

                # wear_prototype_S = (
                #     AGFD_ensemble_emb_dfs[e][
                #         (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
                #         & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
                #         & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                #         & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                #     ][["x", "y"]]
                #     .to_numpy()
                #     .mean(axis=0)
                # )
                # wear_prototype_L = (
                #     AGFD_ensemble_emb_dfs[e][
                #         (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
                #         & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
                #         & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
                #         & (AGFD_ensemble_emb_dfs[e]["load"] != load)
                #     ][["x", "y"]]
                #     .to_numpy()
                #     .mean(axis=0)
                # )

                # Convert severities to target labels [0, 1, 2]
                targets_S = (
                    query_df_S["class"]
                    .map({"healthy": 0, "crack": 1, "wear": 2})
                    .to_numpy()
                )
                targets_L = (
                    query_df_L["class"]
                    .map({"healthy": 0, "crack": 1, "wear": 2})
                    .to_numpy()
                )

                query_embs_S = query_df_S[["x", "y"]].to_numpy()
                query_embs_L = query_df_L[["x", "y"]].to_numpy()

                # query_embs - healthy_prototype_H1
                distances_S = np.stack(
                    [
                        np.linalg.norm(
                            query_embs_S - healthy_prototype_H1[np.newaxis, :],
                            # query_embs - healthy_prototype_H2[np.newaxis, :],
                            axis=1,
                        ),
                        np.linalg.norm(
                            query_embs_S - crack_prototype_L[np.newaxis, :], axis=1
                        ),
                        # np.linalg.norm(
                        #     query_embs_S - wear_prototype_L[np.newaxis, :], axis=1
                        # ),
                    ],
                    axis=1,
                )
                distances_L = np.stack(
                    [
                        np.linalg.norm(
                            query_embs_L - healthy_prototype_H1[np.newaxis, :],
                            # query_embs - healthy_prototype_H2[np.newaxis, :],
                            axis=1,
                        ),
                        np.linalg.norm(
                            query_embs_L - crack_prototype_S[np.newaxis, :], axis=1
                        ),
                        # np.linalg.norm(
                        #     query_embs_L - wear_prototype_S[np.newaxis, :], axis=1
                        # ),
                    ],
                    axis=1,
                )
                ensemble_distances_S.append(distances_S)
                ensemble_distances_L.append(distances_L)

            # Soft voting
            voted_distances_S = np.mean(
                np.stack(ensemble_distances_S, axis=-1), axis=-1
            )
            voted_distances_L = np.mean(
                np.stack(ensemble_distances_L, axis=-1), axis=-1
            )

            acc_S = np.mean(np.argmin(voted_distances_S, axis=1) == targets_S)
            acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
            print(
                f"{speed:<2} Hz, {load:<2}, Nm, installation {installation}: {acc_S:>10.2%} (S), {acc_L:>10.2%} (L)"
            )
            AGFD_accuracies.append((acc_S, acc_L))

print(f"MCC5 average accuracy: {np.mean(AGFD_accuracies):.4%}")

In [ ]:
crack_prototype_L

hp_H1 = pd.DataFrame([healthy_prototype_H1], columns=["x", "y"])
hp_H1["class"] = "healthy"
hp_H1["severity"] = "H1"
hp_H1["role"] = 15

hp_H2 = pd.DataFrame([healthy_prototype_H2], columns=["x", "y"])
hp_H2["class"] = "healthy"
hp_H2["severity"] = "H2"
hp_H2["role"] = 15

cp_S = pd.DataFrame([crack_prototype_S], columns=["x", "y"])
cp_S["class"] = "crack"
cp_S["severity"] = "S"
cp_S["role"] = 15

cp_L = pd.DataFrame([crack_prototype_L], columns=["x", "y"])
cp_L["class"] = "crack"
cp_L["severity"] = "L"
cp_L["role"] = 15

query_S = pd.DataFrame(query_embs_S, columns=["x", "y"])
query_S["class"] = "query_S"
query_S["severity"] = "S"
query_S["role"] = 1

query_L = pd.DataFrame(query_embs_L, columns=["x", "y"])
query_L["class"] = "query_L"
query_L["severity"] = "L"
query_L["role"] = 1

# emb = AGFD_ensemble_emb_dfs[0]
# emb = emb[emb["speed"] == 1000]
# emb = emb[emb["load"] == 6]
# emb = emb[emb["severity"].isin(["H1", "H2", "L"])]

emb = pd.concat([query_S, query_L, hp_H1, hp_H2, cp_L, cp_S], ignore_index=True)

fig = px.scatter(
    emb,
    x="x",
    y="y",
    color="class",
    # symbol="speed",
    # symbol="load",
    symbol="severity",
    size="role",
    # symbol="installation",
    # hover_data=["speed", "load", "class", "severity", "installation"],
    title="AGFD",
)
fig.show()

In [ ]:
# n - 1 installations for prototypes
# 1 installation for query

speed = 1000
load = 6
# e = 0
support_installation = 1

AGFD_accuracies = []

ensemble_distances_S = []
ensemble_distances_L = []

for e in range(5):
    query_df = AGFD_ensemble_emb_dfs[e][
        (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
        & (AGFD_ensemble_emb_dfs[e]["load"] == load)
        & (AGFD_ensemble_emb_dfs[e]["installation"].isin([0, 1, 2]))
        # & (AGFD_ensemble_emb_dfs[e]["severity"] != "H1")
        & (AGFD_ensemble_emb_dfs[e]["severity"].isin(["H2", "L"]))
        & (AGFD_ensemble_emb_dfs[e]["class"].isin(["healthy", "crack"]))
    ]

    # Need separate queries from when M or L are used as the faulty prototype
    # query_df_S = query_df[query_df["severity"] != "L"]
    # query_df_L = query_df[query_df["severity"] != "S"]

    healthy_prototype_H1 = (
        AGFD_ensemble_emb_dfs[e][
            (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
            & (AGFD_ensemble_emb_dfs[e]["severity"] == "H1")
            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
        ][["x", "y"]]
        .to_numpy()
        .mean(axis=0)
    )
    healthy_prototype_H2 = (
        AGFD_ensemble_emb_dfs[e][
            (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
            & (AGFD_ensemble_emb_dfs[e]["severity"] == "H2")
            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
        ][["x", "y"]]
        .to_numpy()
        .mean(axis=0)
    )

    # crack_prototype_S = (
    #     AGFD_ensemble_emb_dfs[e][
    #         (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
    #         & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
    #         & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
    #         & (AGFD_ensemble_emb_dfs[e]["load"] == load)
    #         & (AGFD_ensemble_emb_dfs[e]["installation"] == support_installation)
    #     ][["x", "y"]]
    #     .to_numpy()
    #     .mean(axis=0)
    # )
    crack_prototype_S = (
        AGFD_ensemble_emb_dfs[e][
            (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
            & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
            & (AGFD_ensemble_emb_dfs[e]["installation"] == support_installation)
        ][["x", "y"]]
        .to_numpy()
        .mean(axis=0)
    )
    crack_prototype_L = (
        AGFD_ensemble_emb_dfs[e][
            (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
            & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
            & (AGFD_ensemble_emb_dfs[e]["installation"] != support_installation)
        ][["x", "y"]]
        .to_numpy()
        .mean(axis=0)
    )

    # wear_prototype_S = (
    #     AGFD_ensemble_emb_dfs[e][
    #         (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
    #         & (AGFD_ensemble_emb_dfs[e]["severity"] == "S")
    #         & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
    #         & (AGFD_ensemble_emb_dfs[e]["load"] != load)
    #     ][["x", "y"]]
    #     .to_numpy()
    #     .mean(axis=0)
    # )
    # wear_prototype_L = (
    #     AGFD_ensemble_emb_dfs[e][
    #         (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
    #         & (AGFD_ensemble_emb_dfs[e]["severity"] == "L")
    #         & (AGFD_ensemble_emb_dfs[e]["speed"] != speed)
    #         & (AGFD_ensemble_emb_dfs[e]["load"] != load)
    #     ][["x", "y"]]
    #     .to_numpy()
    #     .mean(axis=0)
    # )

    # Convert severities to target labels [0, 1, 2]
    targets_S = (
        query_df_S["class"].map({"healthy": 0, "crack": 1, "wear": 2}).to_numpy()
    )
    targets_L = (
        query_df_L["class"].map({"healthy": 0, "crack": 1, "wear": 2}).to_numpy()
    )

    query_embs_S = query_df_S[["x", "y"]].to_numpy()
    query_embs_L = query_df_L[["x", "y"]].to_numpy()

    # query_embs - healthy_prototype_H1
    distances_S = np.stack(
        [
            np.linalg.norm(
                query_embs_S - healthy_prototype_H1[np.newaxis, :],
                # query_embs - healthy_prototype_H2[np.newaxis, :],
                axis=1,
            ),
            np.linalg.norm(query_embs_S - crack_prototype_L[np.newaxis, :], axis=1),
            # np.linalg.norm(
            #     query_embs_S - wear_prototype_L[np.newaxis, :], axis=1
            # ),
        ],
        axis=1,
    )
    distances_L = np.stack(
        [
            np.linalg.norm(
                query_embs_L - healthy_prototype_H1[np.newaxis, :],
                # query_embs - healthy_prototype_H2[np.newaxis, :],
                axis=1,
            ),
            np.linalg.norm(query_embs_L - crack_prototype_S[np.newaxis, :], axis=1),
            # np.linalg.norm(
            #     query_embs_L - wear_prototype_S[np.newaxis, :], axis=1
            # ),
        ],
        axis=1,
    )
    ensemble_distances_S.append(distances_S)
    ensemble_distances_L.append(distances_L)

# Soft voting
voted_distances_S = np.mean(np.stack(ensemble_distances_S, axis=-1), axis=-1)
voted_distances_L = np.mean(np.stack(ensemble_distances_L, axis=-1), axis=-1)

acc_S = np.mean(np.argmin(voted_distances_S, axis=1) == targets_S)
acc_L = np.mean(np.argmin(voted_distances_L, axis=1) == targets_L)
print(
    f"{speed:<2} Hz, {load:<2}, Nm, installation {installation}: {acc_S:>10.2%} (S), {acc_L:>10.2%} (L)"
)
AGFD_accuracies.append((acc_S, acc_L))

print(f"MCC5 average accuracy: {np.mean(AGFD_accuracies):.4%}")

In [ ]:
# 1 installation for prototypes
# 1 installation for queries

# speed = 1000
# load = 6
# e = 0
# support_installation = 1
# query_installation = 2
severity = "L"

AGFD_accuracies = []
for support_installation in [1, 2, 3]:
    for query_installation in [1, 2, 3]:
        if support_installation == query_installation:
            continue

        for speed in [500, 750, 1000, 1250, 1500]:
            for load in [1, 6, 11]:
                ensemble_distances = []
                for e in range(5):
                    query_df = AGFD_ensemble_emb_dfs[e][
                        (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                        & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                        & (
                            AGFD_ensemble_emb_dfs[e]["installation"].isin([
                                0,
                                query_installation,
                            ])
                        )
                        & (AGFD_ensemble_emb_dfs[e]["severity"].isin(["H1", severity]))
                        # & (AGFD_ensemble_emb_dfs[e]["class"].isin(["healthy", "crack"]))
                    ]

                    healthy_prototype_H1 = (
                        AGFD_ensemble_emb_dfs[e][
                            (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                            & (AGFD_ensemble_emb_dfs[e]["severity"] == "H1")
                            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                        ][["x", "y"]]
                        .to_numpy()
                        .mean(axis=0)
                    )
                    healthy_prototype_H2 = (
                        AGFD_ensemble_emb_dfs[e][
                            (AGFD_ensemble_emb_dfs[e]["class"] == "healthy")
                            & (AGFD_ensemble_emb_dfs[e]["severity"] == "H2")
                            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                        ][["x", "y"]]
                        .to_numpy()
                        .mean(axis=0)
                    )

                    crack_prototype = (
                        AGFD_ensemble_emb_dfs[e][
                            (AGFD_ensemble_emb_dfs[e]["class"] == "crack")
                            & (AGFD_ensemble_emb_dfs[e]["severity"] == severity)
                            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                            & (
                                AGFD_ensemble_emb_dfs[e]["installation"]
                                == support_installation
                            )
                        ][["x", "y"]]
                        .to_numpy()
                        .mean(axis=0)
                    )

                    wear_prototype = (
                        AGFD_ensemble_emb_dfs[e][
                            (AGFD_ensemble_emb_dfs[e]["class"] == "wear")
                            & (AGFD_ensemble_emb_dfs[e]["severity"] == severity)
                            & (AGFD_ensemble_emb_dfs[e]["speed"] == speed)
                            & (AGFD_ensemble_emb_dfs[e]["load"] == load)
                        ][["x", "y"]]
                        .to_numpy()
                        .mean(axis=0)
                    )

                    # Convert severities to target labels [0, 1, 2]
                    targets = (
                        query_df["class"]
                        .map({"healthy": 0, "crack": 1, "wear": 2})
                        .to_numpy()
                    )

                    query_embs = query_df[["x", "y"]].to_numpy()

                    distances = np.stack(
                        [
                            np.linalg.norm(
                                query_embs - healthy_prototype_H1[np.newaxis, :],
                                # query_embs - healthy_prototype_H2[np.newaxis, :],
                                axis=1,
                            ),
                            np.linalg.norm(
                                query_embs - crack_prototype[np.newaxis, :], axis=1
                            ),
                            np.linalg.norm(
                                query_embs - wear_prototype[np.newaxis, :], axis=1
                            ),
                        ],
                        axis=1,
                    )

                    ensemble_distances.append(distances)

                # Soft voting
                # FIXME
                voted_distances = np.mean(
                    np.stack(ensemble_distances, axis=-1), axis=-1
                )

                acc = np.mean(np.argmin(voted_distances, axis=1) == targets)
                print(
                    f"{speed:<4} RPM, {load / 100:3.0%}, s_inst. {support_installation}, q_inst. {query_installation}: {acc:>10.2%} ({severity})"
                )
                AGFD_accuracies.append((acc))

print(f"AGFD average accuracy: {np.mean(AGFD_accuracies):.4%}")

In [ ]:
# emb = AGFD_ensemble_emb_dfs[0]
emb = query_df

# emb = emb[emb["speed"] == 1000]
# emb = emb[emb["load"] == 6]
emb = emb[emb["installation"].isin([0, 1, 2])]
emb["role"] = 1

healthy_proto = pd.DataFrame([healthy_prototype_H1], columns=["x", "y"])
healthy_proto["class"] = "healthy"
healthy_proto["severity"] = "H1"
healthy_proto["installation"] = 0
healthy_proto["role"] = 15

crack_proto = pd.DataFrame([crack_prototype], columns=["x", "y"])
crack_proto["class"] = "crack"
crack_proto["severity"] = severity
crack_proto["installation"] = support_installation
crack_proto["role"] = 15

wear_proto = pd.DataFrame([wear_prototype], columns=["x", "y"])
wear_proto["class"] = "wear"
wear_proto["severity"] = severity
wear_proto["installation"] = support_installation
wear_proto["role"] = 15

emb = pd.concat([emb, healthy_proto, crack_proto, wear_proto], ignore_index=True)

fig = px.scatter(
    emb,
    x="x",
    y="y",
    color="class",
    # symbol="speed",
    # symbol="load",
    # symbol="severity",
    symbol="installation",
    size="role",
    # hover_data=["speed", "load", "class", "severity", "installation"],
    hover_data=["installation"],
    title="AGFD",
)
fig.show()

## Embeddings plots


In [ ]:
# NOTE: This is for non-ensemble stuff (plotting, etc.)
# Ensemble loads model weights by itself
# model.backbone.load_state_dict(
#     torch.load(
#         model_weight_dir
#         / f"{ensemble_weight_dir_name} R{ensemble_repeat_num}E1"  # Use weights from first ensemble training
#         / f"{ensemble_checkpoint_num}.pth",
#         weights_only=True,
#     )
# )
# model.eval()

# model

In [ ]:
all_emb_dfs = []
for speed in [5, 10, 15, 20]:
    for load in [5, 10, 15]:
        # ANCHOR
        #
        anchor_samples = get_samples_UNSW(UNSW_crack_df, speed, load, "H1")

        # HEALTHY 1
        #

        # Get samples from specific conditions
        healthy_samples_1 = get_samples_UNSW(UNSW_crack_df, speed, load, "H1")
        # Calculate difference against anchor samples
        healthy_diff_1 = (
            healthy_samples_1[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
        )
        # Create embeddings with pretrained model
        healthy_embeddings_1 = embed(healthy_diff_1, model, **embedding_specs)

        # Convert to dataframe and add other necessary info back (class)
        healthy_emb_df_1 = pd.DataFrame(healthy_embeddings_1, columns=embedding_columns)
        healthy_emb_df_1["class"] = "healthy"
        healthy_emb_df_1["severity"] = "H1"

        # HEALTHY 2
        #

        # Get samples from specific conditions
        healthy_samples_2 = get_samples_UNSW(UNSW_crack_df, speed, load, "H2")
        # Calculate difference against anchor samples
        healthy_diff_2 = (
            healthy_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
        )
        # Create embeddings with pretrained model
        healthy_embeddings_2 = embed(healthy_diff_2, model, **embedding_specs)

        # Convert to dataframe and add other necessary info back (class)
        healthy_emb_df_2 = pd.DataFrame(healthy_embeddings_2, columns=embedding_columns)
        healthy_emb_df_2["class"] = "healthy"
        healthy_emb_df_2["severity"] = "H2"

        # Combine all dataframes and add back operating condition info
        emb_df = pd.concat(
            [
                healthy_emb_df_1,
                healthy_emb_df_2,
            ],
            ignore_index=True,
        )
        emb_df["speed"] = speed
        emb_df["load"] = load

        all_emb_dfs.append(emb_df)

UNSW_healthy_emb_dfs = pd.concat(all_emb_dfs, ignore_index=True)

# Plotting
fig = px.scatter(
    UNSW_healthy_emb_dfs,
    x="x",
    y="y",
    color="severity",
    symbol="load",
    hover_data=["class", "speed", "load", "severity"],
    title="UNSW Healthy",
)
fig.show()

In [ ]:
all_emb_dfs = []
for speed in [5, 10, 15, 20]:
    for load in [5, 10, 15]:
        # ANCHOR
        #
        anchor_samples = get_samples_UNSW(UNSW_crack_df, speed, load, "H1")

        # HEALTHY 1
        #

        # Get samples from specific conditions
        crack_samples_1 = get_samples_UNSW(UNSW_crack_df, speed, load, "M")
        # Calculate difference against anchor samples
        crack_diff_1 = (
            crack_samples_1[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
        )
        # Create embeddings with pretrained model
        crack_embeddings_1 = embed(crack_diff_1, model, **embedding_specs)

        # Convert to dataframe and add other necessary info back (class)
        crack_emb_df_1 = pd.DataFrame(crack_embeddings_1, columns=embedding_columns)
        crack_emb_df_1["class"] = "crack"
        crack_emb_df_1["severity"] = "M"

        # crack 2
        #

        # Get samples from specific conditions
        crack_samples_2 = get_samples_UNSW(UNSW_crack_df, speed, load, "L")
        # Calculate difference against anchor samples
        crack_diff_2 = (
            crack_samples_2[:, np.newaxis, :] - anchor_samples[np.newaxis, :, :]
        )
        # Create embeddings with pretrained model
        crack_embeddings_2 = embed(crack_diff_2, model, **embedding_specs)

        # Convert to dataframe and add other necessary info back (class)
        crack_emb_df_2 = pd.DataFrame(crack_embeddings_2, columns=embedding_columns)
        crack_emb_df_2["class"] = "crack"
        crack_emb_df_2["severity"] = "L"

        # Combine all dataframes and add back operating condition info
        emb_df = pd.concat(
            [
                crack_emb_df_1,
                crack_emb_df_2,
            ],
            ignore_index=True,
        )
        emb_df["speed"] = speed
        emb_df["load"] = load

        all_emb_dfs.append(emb_df)

UNSW_crack_emb_dfs = pd.concat(all_emb_dfs, ignore_index=True)

# Plotting
fig = px.scatter(
    UNSW_crack_emb_dfs,
    x="x",
    y="y",
    color="severity",
    symbol="load",
    hover_data=["class", "speed", "load", "severity"],
    title="UNSW Crack",
)
fig.show()

In [ ]:
UNSW_emb_dfs = pd.concat([UNSW_healthy_emb_dfs, UNSW_crack_emb_dfs], ignore_index=True)

# Plotting
fig = px.scatter(
    UNSW_emb_dfs,
    x="x",
    y="y",
    color="class",
    symbol="severity",
    hover_data=["class", "speed", "load", "severity"],
    title="UNSW all [healthy & faulty]",
)
fig.show()

## Results [Non-ensemble]


In [ ]:
# 1 operating condition for prototypes
# n - 1 operating conditions for query

UNSW_accuracies = []
# for speed in [20]:
for speed in [5, 10, 15, 20]:
    for load in [5, 10, 15]:
        prototypes = UNSW_emb_dfs[
            (UNSW_emb_dfs["speed"] == speed) & (UNSW_emb_dfs["load"] == load)
        ]

        healthy_prototype_H1 = (
            prototypes[prototypes["severity"] == "H1"][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )
        healthy_prototype_H2 = (
            prototypes[prototypes["severity"] == "H2"][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )
        crack_prototype_M = (
            prototypes[prototypes["severity"] == "M"][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )
        crack_prototype_L = (
            prototypes[prototypes["severity"] == "L"][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )

        query_df = UNSW_emb_dfs[
            ~((UNSW_emb_dfs["speed"] == speed) & (UNSW_emb_dfs["load"] == load))
            & (UNSW_emb_dfs["severity"] != "H1")
        ]

        targets = query_df["severity"].map({"H2": 0, "M": 1, "L": 1}).to_numpy()
        query_embs = query_df[["x", "y"]].to_numpy()

        # query_embs - healthy_prototype_H1
        distances = np.stack(
            [
                np.linalg.norm(
                    query_embs - healthy_prototype_H1[np.newaxis, :],
                    # query_embs - healthy_prototype_H2[np.newaxis, :],
                    axis=1,
                ),
                np.linalg.norm(query_embs - crack_prototype_M[np.newaxis, :], axis=1),
                np.linalg.norm(query_embs - crack_prototype_L[np.newaxis, :], axis=1),
            ],
            axis=1,
        )
        acc_M = np.mean(np.argmin(distances[:, [0, 1]], axis=1) == targets)
        acc_L = np.mean(np.argmin(distances[:, [0, 2]], axis=1) == targets)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_M:>10.2%} (M), {acc_L:>10.2%} (L)")
        UNSW_accuracies.append((acc_M, acc_L))

print(f"UNSW Average Accuracy: {np.mean(UNSW_accuracies):.2%}")

In [ ]:
# n - 1 operating conditions for prototypes
# 1 operating condition for query

UNSW_accuracies = []
for speed in [5, 10, 15, 20]:
    for load in [5, 10, 15]:
        query_df = UNSW_emb_dfs[
            (UNSW_emb_dfs["speed"] == speed)
            & (UNSW_emb_dfs["load"] == load)
            & (UNSW_emb_dfs["severity"] != "H1")
        ]

        healthy_prototype_H1 = (
            UNSW_emb_dfs[
                (UNSW_emb_dfs["severity"] == "H1")
                & (UNSW_emb_dfs["speed"] != speed)
                & (UNSW_emb_dfs["load"] != load)
            ][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )
        healthy_prototype_H2 = (
            UNSW_emb_dfs[
                (UNSW_emb_dfs["severity"] == "H2")
                & (UNSW_emb_dfs["speed"] != speed)
                & (UNSW_emb_dfs["load"] != load)
            ][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )
        crack_prototype_M = (
            UNSW_emb_dfs[
                (UNSW_emb_dfs["severity"] == "M")
                & (UNSW_emb_dfs["speed"] != speed)
                & (UNSW_emb_dfs["load"] != load)
            ][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )
        crack_prototype_L = (
            UNSW_emb_dfs[
                (UNSW_emb_dfs["severity"] == "L")
                & (UNSW_emb_dfs["speed"] != speed)
                & (UNSW_emb_dfs["load"] != load)
            ][["x", "y"]]
            .to_numpy()
            .mean(axis=0)
        )

        targets = (
            query_df["severity"].map({"H1": 0, "H2": 0, "M": 1, "L": 1}).to_numpy()
        )
        query_embs = query_df[["x", "y"]].to_numpy()

        # query_embs - healthy_prototype_H1
        distances = np.stack(
            [
                np.linalg.norm(
                    query_embs - healthy_prototype_H1[np.newaxis, :],
                    # query_embs - healthy_prototype_H2[np.newaxis, :],
                    axis=1,
                ),
                np.linalg.norm(query_embs - crack_prototype_M[np.newaxis, :], axis=1),
                np.linalg.norm(query_embs - crack_prototype_L[np.newaxis, :], axis=1),
            ],
            axis=1,
        )
        acc_M = np.mean(np.argmin(distances[:, [0, 1]], axis=1) == targets)
        acc_L = np.mean(np.argmin(distances[:, [0, 2]], axis=1) == targets)
        print(f"{speed:<2} Hz, {load:<2} Nm: {acc_M:>10.2%} (M), {acc_L:>10.2%} (L)")
        UNSW_accuracies.append((acc_M, acc_L))

print(f"UNSW Average Accuracy: {np.mean(UNSW_accuracies):.2%}")